# Multi-Modal Genre Classification Ensemble & Track-Level Voting

This notebook unites all trained models into a single high-accuracy ensemble:
1. **Tabular Feature Models**: Support Vector Machine (RBF), Random Forest, and PyTorch MLP.
2. **Spectrogram CNN Models**: ResNet-18, EfficientNet-B0, and MobileNetV2.

### Ensembling Strategies
- **Weighted Soft Voting**: Fuses predicted probability distributions $P = \sum w_i P_i$.
- **Track-Level (Full Song) Aggregation**: Averages the probability predictions of all 10-second segments belonging to the same song (`track_id`), achieving massive accuracy gains (typically +5% to +8%) over single-segment evaluation.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn.functional as F

BASE_DIR = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
MODELS_DIR = BASE_DIR / "models"
FEATURES_DIR = MODELS_DIR / "features"
MELS_DIR = MODELS_DIR / "mels"

print(f"Project Base: {BASE_DIR}")
print(f"Features Models Dir: {FEATURES_DIR}")
print(f"Mels Models Dir: {MELS_DIR}")

In [ ]:
def load_or_generate_probabilities():
    """
    Loads saved test probability arrays from all trained models.
    """
    model_probs = {}
    
    # Check for CNN probability files
    for cnn_name in ["resnet18", "efficientnet", "mobilenetv2"]:
        path = MELS_DIR / f"{cnn_name}_test_probs.npy"
        if not path.exists():
            path = Path(f"{cnn_name}_test_probs.npy")
        if path.exists():
            model_probs[cnn_name] = np.load(path)
            print(f"Loaded {cnn_name} test probabilities: shape {model_probs[cnn_name].shape}")
        else:
            print(f"Note: {path.name} not found yet (run {cnn_name}.ipynb evaluation cell first).")
            
    # Check for Tabular probability files
    for tab_name in ["rf", "svm", "mlp"]:
        path = FEATURES_DIR / f"{tab_name}_test_probs.npy"
        if path.exists():
            model_probs[tab_name] = np.load(path)
            print(f"Loaded {tab_name} test probabilities: shape {model_probs[tab_name].shape}")
        else:
            print(f"Note: {path.name} not found yet (run {tab_name} notebook evaluation cell first).")
            
    # Load ground truth labels and track IDs
    labels_path = MELS_DIR / "resnet18_test_labels.npy"
    if not labels_path.exists():
        labels_path = Path("resnet18_test_labels.npy")
        
    tracks_path = MELS_DIR / "resnet18_test_tracks.npy"
    if not tracks_path.exists():
        tracks_path = Path("resnet18_test_tracks.npy")
        
    y_true = np.load(labels_path) if labels_path.exists() else None
    track_ids = np.load(tracks_path) if tracks_path.exists() else None
    
    return model_probs, y_true, track_ids

model_probs, y_true, track_ids = load_or_generate_probabilities()

In [ ]:
def run_ensemble(model_probs, y_true, track_ids=None, weights=None, classes=None):
    """
    Computes weighted ensemble probabilities and track-level voting.
    """
    if not model_probs:
        print("No model probabilities available to ensemble.")
        return None
        
    available_models = list(model_probs.keys())
    print(f"Available models for ensemble: {available_models}")
    
    # Default heuristic weights prioritizing strongest models
    default_weight_map = {
        "resnet18": 0.35,
        "efficientnet": 0.30,
        "mobilenetv2": 0.15,
        "rf": 0.08,
        "svm": 0.06,
        "mlp": 0.06
    }
    
    if weights is None:
        raw_weights = [default_weight_map.get(m, 0.1) for m in available_models]
        weights = np.array(raw_weights) / sum(raw_weights)
        
    print("\nEnsemble Model Weights:")
    for m, w in zip(available_models, weights):
        print(f"  - {m:15s}: {w * 100:.1f}%")
        
    # Weighted average of probability distributions
    p_ensemble = np.zeros_like(next(iter(model_probs.values())))
    for m, w in zip(available_models, weights):
        p_ensemble += w * model_probs[m]
        
    seg_preds = np.argmax(p_ensemble, axis=1)
    
    if y_true is not None:
        seg_acc = accuracy_score(y_true, seg_preds)
        print(f"\n==========================================")
        print(f"Segment-Level Ensemble Accuracy: {seg_acc * 100:.2f}%")
        print(f"==========================================")
        
        # Track-Level (Full Song) Evaluation
        if track_ids is not None:
            df_ensemble = pd.DataFrame(p_ensemble)
            df_ensemble['track_id'] = track_ids
            df_ensemble['true_label'] = y_true
            
            # Average probabilities across all 10-second segments of each song
            track_probs = df_ensemble.groupby('track_id').mean()
            track_y_true = df_ensemble.groupby('track_id')['true_label'].first()
            track_preds = track_probs.drop(columns=['true_label']).values.argmax(axis=1)
            track_acc = accuracy_score(track_y_true, track_preds)
            
            print(f"Track-Level (Full Song) Ensemble Accuracy: {track_acc * 100:.2f}%")
            print(f"==========================================")
            
        if classes is not None:
            print("\nClassification Report (Segment-Level):")
            print(classification_report(y_true, seg_preds, target_names=classes))
            
    return p_ensemble

# Run ensemble if data loaded
classes_path = MELS_DIR / "resnet18_classes.npy"
classes = np.load(classes_path) if classes_path.exists() else None
p_ensemble = run_ensemble(model_probs, y_true, track_ids, classes=classes)

In [ ]:
if y_true is not None and p_ensemble is not None:
    seg_preds = np.argmax(p_ensemble, axis=1)
    cm = confusion_matrix(y_true, seg_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes if classes is not None else range(9),
                yticklabels=classes if classes is not None else range(9))
    plt.xlabel('Predicted Genre')
    plt.ylabel('True Genre')
    plt.title('Multi-Modal Ensemble Confusion Matrix')
    plt.show()